In [2]:
import math
from typing import List, Tuple, Optional

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, Polygon


def _read_patch_and_transform(ortho_path: str, geom, pad_pixels: int = 10) -> Tuple[Optional[np.ndarray], Optional[object], Optional[Polygon]]:
    """Read a cropped orthomosaic patch for the provided geometry using rasterio.mask.mask.
code
python
# Demo cell: find a split/connected crown node and show its orthomosaic patches
import sys
from pprint import pprint

# Try to pick a tracker object created earlier in the notebook
tracker = None
candidate_names = [
    'iou_one_to_one_summary', 'iou_one_to_one',
    'trade_summary', 'trade_tracker',
    'partial_trade_v2_summary', 'partial_trade_v2_tracker',
    'tracker_strict', 'tracker_balanced', 'tracker_threshold',
    'viz_tracker', 'tracker'
]
for name in candidate_names:
    if name in globals():
        val = globals()[name]
        # If it's a summary dict with a 'tracker' entry, prefer that
        if isinstance(val, dict) and 'tracker' in val:
            tracker = val['tracker']
            print(f"Using tracker from summary variable: {name}")
            break
        # Otherwise, if it looks like a tracker object (has .G), use it
        try:
            if hasattr(val, 'G'):
                tracker = val
                print(f"Using tracker variable: {name}")
                break
        except Exception:
            pass

if tracker is None:
    raise RuntimeError(
        "No tracker object found in the notebook globals. Run one of the experiment cells that produces a tracker (e.g. the cells that set iou_one_to_one_summary, trade_summary, partial_trade_v2_summary, etc.)."
    )

# Import the visualization helpers you already added
try:
    from src.utils.crown_vis import find_a_split_node, show_connected_crown_patches
except Exception as e:
    # Try alternate import path if notebook root differs
    try:
        import importlib.util, pathlib
        vis_path = pathlib.Path('.') / 'src' / 'utils' / 'crown_vis.py'
        vis_path = vis_path.resolve()
        if vis_path.exists():
            spec = importlib.util.spec_from_file_location('crown_vis', str(vis_path))
            crown_vis = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(crown_vis)
            find_a_split_node = crown_vis.find_a_split_node
            show_connected_crown_patches = crown_vis.show_connected_crown_patches
        else:
            raise
    except Exception:
        raise

# Find a candidate node having 2+ split successors
node = None
try:
    node = find_a_split_node(tracker)
    if node is not None:
        print('Found split node:', node)
except Exception as e:
    print('Error while searching for split node:', e)

# Fallback: pick any node that has at least one successor
if node is None:
    for n in tracker.G.nodes:
        if list(tracker.G.successors(n)):
            node = n
            print('Fallback selected node (has successors):', node)
            break

if node is None:
    raise RuntimeError('Could not find any node with successors in the tracker graph. Build the graph first.')

# Display the query crown plus connected successors (side-by-side)
show_connected_crown_patches(
    tracker,
    node,
    include_prev=True,
    show_all_successors=True,
    max_successors=6,
    figsize_per_patch=(5, 5),
)

    Returns (img_hwc (uint8 or float), out_transform, geometry) or (None, None, None) on failure.
    """
    if ortho_path is None:
        return None, None, None
    try:
        with rasterio.open(ortho_path) as src:
            # Ensure geometry mapping
            geom_mapping = [mapping(geom)]
            out_image, out_transform = mask(src, geom_mapping, crop=True)
            # out_image: (bands, H, W)
            img = np.moveaxis(out_image, 0, -1)
            # Normalize to uint8 for display if needed
            if img.dtype != np.uint8:
                try:
                    img_min = float(img.min())
                    img_ptp = float(img.ptp())
                    if img_ptp == 0:
                        img = (img - img_min).astype(np.uint8)
                    else:
                        img = ((img - img_min) / (img_ptp) * 255.0).astype(np.uint8)
                except Exception:
                    pass
            return img, out_transform, geom
    except Exception as e:
        print(f"Could not read patch from {ortho_path}: {e}")
        return None, None, None


def _geom_to_pixel_coords(geom, out_transform) -> Tuple[np.ndarray, np.ndarray]:
    """Convert a Shapely polygon geometry into pixel coordinates relative to the cropped patch using the out_transform returned by rasterio.mask.mask."""
    if not isinstance(geom, Polygon):
        # fallback: attempt to use exterior if available
        try:
            geom = Polygon(geom)
        except Exception:
            return np.array([]), np.array([])
    x_geo, y_geo = geom.exterior.xy
    # rasterio.transform.rowcol expects (transform, xs, ys) in geographic units -> returns (rows, cols)
    try:
        from rasterio.transform import rowcol

        rows, cols = rowcol(out_transform, x_geo, y_geo)
        xs = np.array(cols)
        ys = np.array(rows)
        return xs, ys
    except Exception:
        # Best-effort fallback: return empty arrays
        return np.array([]), np.array([])


def show_connected_crown_patches(
    tracker,
    node: Tuple[int, int],
    include_prev: bool = True,
    show_all_successors: bool = True,
    max_successors: int = 6,
    figsize_per_patch: Tuple[int, int] = (5, 5),
):
    """Visualize the orthomosaic patch for a given crown node and its connected crowns.

    Args:
        tracker: TreeTrackingGraph instance (from the notebook). The tracker must have discovered files (tracker.file_pairs and tracker.om_ids).
        node: Tuple[om_id, crown_id] selecting the crown of interest.
        include_prev: if True, include the input node's patch as the first panel.
        show_all_successors: if True include all successors of the node (filtered by graph edges). Otherwise include only successors with case 'split' or 'merge'.
        max_successors: cap number of successor patches shown to avoid huge figures.
        figsize_per_patch: size per subplot (width, height)

    Behavior:
        - For each selected node (prev + successors), the function reads the orthomosaic with rasterio.mask.mask cropping to the polygon geometry.
        - It then displays the image patch and overlays the polygon outline in red (prev) or orange/green (successors).

    Notes:
        - This is intended for interactive notebook use. It opens matplotlib figures inline.
    """
    if tracker is None:
        raise ValueError("tracker must be provided")
    if not hasattr(tracker, 'file_pairs') or not tracker.file_pairs:
        # Attempt to discover files
        tracker.discover_files()

    om_id, crown_id = node

    if om_id not in tracker.om_ids:
        raise ValueError(f"OM id {om_id} not found in tracker.om_ids")

    # Build list of nodes to visualize
    panels: List[Tuple[Tuple[int, int], str]] = []  # ((om, cid), role)
    if include_prev:
        panels.append((node, 'prev'))

    succs = list(tracker.G.successors(node)) if node in tracker.G else []
    if not succs:
        print(f"No successors found for node {node}")
    else:
        if show_all_successors:
            chosen = succs[:max_successors]
        else:
            # Filter by split/merge cases
            chosen = [s for s in succs if tracker.G[node][s].get('case') in ('split', 'merge')][:max_successors]
        for s in chosen:
            panels.append((s, 'succ'))

    if not panels:
        print("No panels to display.")
        return

    n_panels = len(panels)
    fig_w = figsize_per_patch[0] * n_panels
    fig_h = figsize_per_patch[1]
    fig, axes = plt.subplots(1, n_panels, figsize=(fig_w, fig_h))
    if n_panels == 1:
        axes = [axes]

    for ax, (pnode, role) in zip(axes, panels):
        pom, pcid = pnode
        # Resolve ortho file for this OM
        try:
            idx = tracker.om_ids.index(pom)
            crown_file, ortho_file = tracker.file_pairs[idx]
        except Exception:
            ortho_file = None

        # Get geometry for polygon
        try:
            gdf = tracker.crowns_gdfs[pom]
            geom = gdf.iloc[pcid].geometry
        except Exception:
            geom = None

        img, out_transform, geom_used = (None, None, None)
        if ortho_file and geom is not None:
            img, out_transform, geom_used = _read_patch_and_transform(ortho_file, geom)

        if img is None:
            ax.text(0.5, 0.5, f'No image for OM {pom} Crown {pcid}', va='center', ha='center')
            ax.set_title(f'OM {pom} Crown {pcid} (no image)')
            ax.axis('off')
            continue

        ax.imshow(img)
        title = f'OM {pom} Crown {pcid}'
        if role == 'prev':
            title += ' (query)'
            edgecol = 'red'
        else:
            edgecol = 'orange'
        ax.set_title(title)
        ax.axis('off')

        # Overlay polygon outline in pixel coords
        try:
            xs, ys = _geom_to_pixel_coords(geom_used, out_transform)
            if xs.size and ys.size:
                ax.plot(xs, ys, linewidth=2, color=edgecol)
        except Exception:
            pass

    plt.tight_layout()
    plt.show()


# Small convenience utility for notebook: find a sample node with split successors
def find_a_split_node(tracker) -> Optional[Tuple[int, int]]:
    """Return any node (om_id, crown_id) that has 2+ successors classified as 'split'."""
    for node in tracker.G.nodes:
        succs = list(tracker.G.successors(node))
        split_succs = [s for s in succs if tracker.G[node][s].get('case') == 'split']
        if len(split_succs) >= 2:
            return node
    return None
